In [ ]:
# @title 1. 本のp.181 (6-1-1) ultralyticsインストール

# 本ではWindowsのコマンドプロンプトで実行しているが、Colabでは先頭に ! を足す

!pip install ultralytics

# 補足
# - 実行するとインストール中の表示がたくさん出る (うざい)
# - Colab特有の機能として「!」で始まる行は「シェルコマンド」として扱われる
# - もはやPythonと無関係 (先頭に!を付ける文法はPythonに無い)
# - CoalbではOpenCVのようにインストール不要で使えるライブラリも多数ある


In [ ]:
# @title 2. インストール中の表示を消す方法

!pip install ultralytics > /dev/null


In [ ]:
# @title 3. インストールコマンドをちゃんとPythonで書く

import subprocess
subprocess.run(["pip", "install", "ultralytics"])

# 個人的にはこれがお勧め。理由は
# - 素のPythonプログラムとして動く
# - ipynbファイルとしてGitHubに置いて表示した時、文字化けしない


In [ ]:
# @title 4. 本のp.182 (6-1-2) 物体検出ライブラリを使えることの確認

from ultralytics import YOLO

# モデルを設定
model = YOLO('yolov8n.pt')


In [ ]:
# @title 5. 動画撮影用の独自クラス ColabCap
# 本の cap = cv2.VideoCapture(0) の代用
# Chapter 5 で作ったものと同一
# https://github.com/ec22s/colab-ikinari-python/blob/main/chapter-5/ColabCap.py

from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np
import cv2

class ColabCap:

  _js = '''
    let video = document.createElement('video');
    let canvas = document.createElement('canvas');
    let stream = null;

    async function createDom() {
      if (stream) return;
      stream = await navigator.mediaDevices.getUserMedia({ video: true });
      video.srcObject = stream;
      await video.play();
    }

    async function removeDom() {
      await stream.getVideoTracks()[0].stop();
      video = null;
      stream = null;
      canvas = null;
    }

    async function cap(quality, waitSec) {
      if (!stream) await createDom();
      await new Promise((resolve) => setTimeout(resolve, waitSec * 10**3));
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      return canvas.toDataURL('image/jpeg', quality);
    }
  '''

  def __init__(self, quality=0.8, first_wait_sec=0.25):
    self.quality = quality
    self.first_wait_sec = first_wait_sec
    display(Javascript(ColabCap._js))

  def read(self):
    try:
      data = eval_js(f'cap({ self.quality }, { self.first_wait_sec })')
      self.first_wait_sec = 0
      image_bytes = b64decode(data.split(',')[1])
      jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
      return True, cv2.imdecode(jpg_as_np, flags=1)
    except Exception as err:
      print(str(err))
      return False, None

  def release(self):
    eval_js('removeDom()')


In [ ]:
# @title 6. 本のp.184 (6-2-1) のように動画撮影＆1フレームずつ表示 (ただしゆっくり)

# 未実行なら事前に実行するセル：5

# ネット越し＆Colabを使うので速い撮影はできず、撮影条件を調整する
# 開始して動画が出るまで少し待つ
# 初回実行時は
# - 最初のカメラ利用確認で許可を選ぶ。この時はエラーで終わる
# - もう一度実行すると別のカメラ利用確認が出るのでそれも許可する
# - これで正常に動くはず

import cv2
import time
from google.colab.patches import cv2_imshow
from google.colab import output

# カメラの初期化
cap = ColabCap()

# 撮影条件
duration = 30 # 撮影する秒数
frame_rate = 0.2 # 5秒間に1フレーム

# 動画撮影
interval = 1 / frame_rate
frame_count = int(duration / interval)
for i in range(frame_count):
  ret, frame = cap.read()
  if not ret:
    break

  # 画像表示
  output.clear()
  cv2_imshow(frame)
  time.sleep(interval)


In [ ]:
# @title 7. 本のp.185 (6-2-2) のように動画撮影中に物体検出 (ただしゆっくり)

# 前セルに追加した行：5, 13, 31, 32
# 未実行なら事前に実行するセル：3, 5

from ultralytics import YOLO
import cv2
import time
from google.colab.patches import cv2_imshow
from google.colab import output

# モデルを設定
model = YOLO('yolov8n.pt')

# カメラの初期化
cap = ColabCap()

# 撮影条件
duration = 30 # 撮影する秒数
frame_rate = 0.2 # 5秒間に1フレーム

# 動画撮影
interval = 1 / frame_rate
frame_count = int(duration / interval)
for i in range(frame_count):
  ret, frame = cap.read()
  if not ret:
    break

  # 物体検出
  results = model(frame, verbose=False)
  img_annotated = results[0].plot()

  # 画像表示
  output.clear()
  cv2_imshow(img_annotated)
  time.sleep(interval)

# 成功すれば色々試せる → 本p.185〜187参照
# 今回 (#13, 6/5) はここまで
